In [ ]:
# Linear algebra support
import numpy as np
import scipy.optimize

# Simulation engine
import espressomd
import espressomd.observables
import espressomd.accumulators
import espressomd.analyze

# Other packages
import matplotlib.pyplot as plt
from dataclasses import dataclass
from rich.progress import track
import h5py as hf

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for storing simulation parameters.
    
    Attributes
    ----------
    temperature : float
        Temperature of the simulation.
    pressure : float
        Pressure of the system.
    number_of_particles : int
        Number of particles in the box.
    simulation_time : int
        How long the simulation should run for.
    """
    # Box setup
    temperature: float
    pressure: float
    density: float
    number_of_particles: int
        
    # Simultaion parameters
    simulation_time: int
    measurement_interval: int
    time_step: float
    skin: float = 0.4
    gamma_t: float = 1.0
    gamma_v: float = 1.0
        
    # Interaction parameters
    sigma: float = 1.0
    epsilon: float = 1.0
    cutoff: float = 2.5
        
    # Geometry optimization
    total_force: float = 1e-2
    damping: int = 30
    max_steps: int = 10000
    max_displacement: float = 0.01 * 1.0
    em_step: int = 10


In [ ]:
def write_database(colloid_data: np.ndarray):
    """
    Write data to the database
    """

In [ ]:
def perform_simulation(parameters: SimulationParameters):
    """
    Function to perform a full simulation
    """
    # Side length of the box
    side_length = np.power(
        parameters.number_of_particles / parameters.density, 1.0 / 3.0
    ) * np.ones(3)
    
    # Create the espresso system
    system = espressomd.System(box_l=side_length)
    system.time_step = parameters.time_step
    system.cell_system.skin = parameters.skin
    
    # Add the particles to the system
    particles = system.part.add(
        type=[0] * parameters.number_of_particles, 
        pos=np.random.random((parameters.number_of_particles, 3)) * system.box_l
    )
    
    # Set interaction parameters
    system.non_bonded_inter[0, 0].lennard_jones.set_params(
        epsilon=parameters.epsilon, 
        sigma=parameters.sigma, 
        cutoff=parameters.cutoff, 
        shift=0
    )
    
    # Perform geometry optimization
    
    # Set up steepest descent integration
    system.integrator.set_steepest_descent(
        f_max=0,
        gamma=parameters.damping, 
        max_displacement=parameters.max_displacement
    )

    # Initialize integrator to obtain initial forces
    system.integrator.run(0)
    old_force = np.max(
        np.linalg.norm(system.part.all().f, axis=1)
    )

    while system.time / system.time_step < parameters.max_steps:
        system.integrator.run(parameters.em_step)
        force = np.max(np.linalg.norm(system.part.all().f, axis=1))
        rel_force = np.abs((force - old_force) / old_force)

        if rel_force < parameters.total_force:
            break
        old_force = force
        
    # Perform NPT simulations
    
    system.time = 0.

    system.thermostat.set_npt(
        kT=parameters.temperature, 
        gamma0=parameters.gamma_t, 
        gammav=parameters.gamma_v,
        seed=np.random.randint(876)
    )
    system.integrator.set_isotropic_npt(
        ext_pressure=parameters.pressure, piston=1.0
    )

    for i in track(range(
        int(parameters.simulation_time / parameters.measurement_interval)
    ), description="NPT Simulation"):
        system.integrator.run(parameters.measurement_interval)
        
    # Perform NVT simulation
    
    system.time = 0.
    system.thermostat.turn_off()
    
    system.integrator.set_vv()
    system.thermostat.set_langevin(
        kT=parameters.temperature, 
        gamma=parameters.gamma_t, 
        seed=np.random.randint(434)
    )
    
    for i in track(range(
        int(parameters.simulation_time / parameters.measurement_interval)
    ), description="NVT Simulation"):
        system.integrator.run(parameters.measurement_interval)
        
    return system


In [ ]:
parameters = SimulationParameters(
    temperature=50.0,
    pressure=1.0,
    density=1.0,
    number_of_particles=500,
    simulation_time=10000,
    measurement_interval=1000,
    time_step=1e-3
)

In [ ]:
final_system = perform_simulation(parameters)